# Transformers

In this notebook, we explore the Transformer architecture, the main neural network architecture behind modern language and vision models, and use it for two tasks: text classification and image-text understanding.

We start by introducing the Transformer, then look at attention in code, then use a pre-trained BERT-like model to classify texts. We close by briefly recapping Convolutional Neural Networks and moving to CLIP, a transformer-based model that connects images and text.

## The Transformer

Published in the paper [Attention is All you Need](https://arxiv.org/abs/1706.03762) by Vaswani et al. (2017), the Transformer and its variants have become the main neural network architecture across data modalities. The Transformer is [natively implemented in PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html) as a network model.

In our introduction to this architecture, we borrow from the following sources:
1. [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/).
2. [A Very Gentle Introduction to Large Language Models without the Hype](https://mark-riedl.medium.com/a-very-gentle-introduction-to-large-language-models-without-the-hype-5f67941fa59e).
3. [Tutorial 6: Transformers and Multi-Head Attention](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html).
4. [The Illustrated GPT2](https://jalammar.github.io/illustrated-gpt2/).


### Language Models

**Language models** allow us to represent textual inputs numerically, all the while storing information about all layers of the linguistic stack, from syntax to semantics, and more.

Every language model starts with the need to represent text numerically. This step is often called **tokenization**: the slicing up of a text into units that are assigned a vector representation, or **embedding**, before being further processed by the language model. The most widely used tokenizers nowadays work at the subword level, i.e., short sequences of characters. Many different alternatives are possible.

Once a text has been tokenized, and in order to train a language model, an approach is taken which is called **self-supervision**: large quantities of human-generated texts are used with words masked at random or sequentially, and the model is trained on the task of predicting the missing words. This simple approach turns out to be extremely powerful in practice. See [2] for more.

Random masking:

<img src="figures/mask1.png" width="400px" heigth="400px">

Sequential masking, or next word prediction:

<img src="figures/mask2.png" width="400px" heigth="400px">

"GPT stands for **Generative Pre-trained Transformer**. Let’s break this down:

* Generative. The model is capable of generating continuations to the provided input. That is, given some text, the model tries to guess which words come next.
* Pre-trained. The model is trained on a very large corpus of general text and is meant to be trained once and used for a lot of different things without needing to be re-trained from scratch.

A **transformer** is a particular type of deep learning model that transforms the encoding in a particular way that makes it easier to guess the blanked out word. At the heart of a transformer is the classical **encoder-decoder network**. The encoder does a very standard encoding process, but then it adds something else called **self-attention**.

A transformer learns which words in an input sequence are related and then creates a new encoding for each position in the input sequence that is a merger of all the related words." [2]

<img src="figures/transformer_architecture.svg" width="400px" heigth="800px">

This figure illustrates a full encoder-decoder architecture, typically adopted for tasks where the full input needs to be accounted for when generating an output. An example is machine translation, or modality change (e.g., speech to text).

### Encoder-Decoder Architectures

<img src="figures/encoderdecoder1.png" width="400px" heigth="400px">

The basic idea of Encoder-Decoder architectures originates by the need to deal with so called *"sequence to sequence tasks"*, e.g., translation. The input, for example your text in French, is first *encoded*, and then the output, for example a translation in English, is generated by the *decoder* on the basis of the encoded input. It turns out that these architectures are also generally useful when you want to generate an answer from a query, like ChatGPT does.

<img src="figures/encoderdecoder3.png" width="500px" heigth="300px">

In the encoder and the decoder, we can use a variety of modules. 

With transformers, feed forward layers are used in conjunction with self-attention layers. When the encoded input is passed to the decoder, another form of attention is applied: cross-attention:

<img src="figures/encoderdecoder2.png" width="500px" heigth="300px">

In cross-attention, the queries (*Q*) come from the decoder hidden states, while the keys (*K*) and values (*V*) come from the encoder hidden states. We will see next what this means.

### Attention

<img src="figures/attention1.png" width="500px" heigth="400px">

The key idea of attention is to allow the model to look at, or attend, at any relevant part of the input when making a prediction, for example when predicting a masked word, or translating a single word into another language.

<img src="figures/attention2.png" width="400px" heigth="400px">

Attention units can be visualized and show how a model might consider different parts of the input at a given step, with varied intensity based on how relevant each part of the input is to the prediction at this step.

#### Self-Attention

Attention units work by introducing three sets of parameters: *queries Q, keys K, and values V*.

Following [1], we have that at each calculation of self-attention:

1. The first step is to create three vectors from each of the encoder’s input vectors (in this case, the embedding of each word). So for each word, we create a query vector, a key vector, and a value vector. These vectors are created by multiplying the embedding by the three matrices Q, K, V, that are trained during the training process.
2. The second step in calculating self-attention is to calculate a score. Say we’re calculating the self-attention for the first word in this example, “Thinking”. We need to score each word of the input sentence against this word. The score determines how much focus to place on other parts of the input sentence as we encode a word at a certain position.
3. The third step applies a normalization and the softmax, to get probability scores for each input word against the current word.
4. The next step is to multiply each value vector by the softmax score (in preparation to sum them up). The intuition here is to keep intact the values of the word(s) we want to focus on, and reduce the weight of irrelevant words.
5. The fifth step is to sum up the weighted value vectors. This produces the output of the self-attention layer at this position (for the first word).

<img src="figures/selfattention.png" width="500px" heigth="800px">

These operations are easily implemented as matrix multiplications. 

Note again that **cross-attention** works in a similar way, just the queries come from the decoder, whilst the keys and values from the encoder. Lastly, note that the transformer used **multi-head attention**: several attentio modules or heads are stacked up in each layer, the model training determine how they are actually used. This gives the model much more capacity.

*See 6.2. How Does Self-Attention Work? [2] and [1] here for an in-depth discussion.*

#### Attention in code

Let's implement scaled dot-product self-attention from scratch, on a toy example, to make the steps above concrete.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme('notebook', style='whitegrid')

torch.manual_seed(42)

tokens = ["the", "cat", "sat", "on", "the", "mat"]
seq_len, d_model = len(tokens), 8

# A toy input: one random embedding vector per token
x = torch.randn(seq_len, d_model)

# The three learned projections (queries, keys, values)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

Q = x @ W_q
K = x @ W_k
V = x @ W_v

# Step 2-3: scores, scaled and normalized with softmax
scores = (Q @ K.T) / (d_model ** 0.5)
attention_weights = F.softmax(scores, dim=-1)

# Step 4-5: weighted sum of values
output = attention_weights @ V

print("Attention weights (rows sum to 1):")
print(attention_weights.round(decimals=2))

plt.figure(figsize=(5, 4))
sns.heatmap(attention_weights.detach().numpy(), xticklabels=tokens, yticklabels=tokens,
            cmap="viridis", annot=True, fmt=".2f", cbar=True)
plt.title("Self-attention weights (toy example)")
plt.xlabel("attending to")
plt.ylabel("query token")
plt.show()

In [ ]:
import torch.nn as nn

# The same computation with PyTorch's built-in multi-head attention (num_heads=1 to match the toy example)
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=1, batch_first=True)
mha_output, mha_weights = mha(x.unsqueeze(0), x.unsqueeze(0), x.unsqueeze(0))
print("nn.MultiheadAttention output shape:", mha_output.shape)

# Causal masking: a query can only attend to itself and earlier tokens (used in decoder-only, GPT-style models)
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
causal_attention_weights = F.softmax(masked_scores, dim=-1)

plt.figure(figsize=(5, 4))
sns.heatmap(causal_attention_weights.detach().numpy(), xticklabels=tokens, yticklabels=tokens,
            cmap="viridis", annot=True, fmt=".2f", cbar=True)
plt.title("Causal (masked) self-attention: each token only sees itself and the past")
plt.xlabel("attending to")
plt.ylabel("query token")
plt.show()

#### Attention in a real model

Let's look at the attention patterns of a real pre-trained model, `bert-base-uncased`, on an actual sentence.

In [ ]:
from transformers import BertModel, BertTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", clean_up_tokenization_spaces=True)
bert_model = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_model.eval()

sentence = "The council met to discuss the new railway station."
inputs = bert_tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = bert_model(**inputs)

# outputs.attentions: one tensor per layer, shape (batch, num_heads, seq_len, seq_len)
last_layer_attention = outputs.attentions[-1][0]  # last layer, drop batch dim
head_0_attention = last_layer_attention[0]  # first head

bert_tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

plt.figure(figsize=(7, 6))
sns.heatmap(head_0_attention.numpy(), xticklabels=bert_tokens, yticklabels=bert_tokens, cmap="viridis")
plt.title("BERT last-layer, head 0 self-attention")
plt.xticks(rotation=90)
plt.show()

Note that the transformer is a general purpose architecture. It has been used in encoder-only architectures (e.g., BERT: Bidirectional Encoder Representations from Transformers), decoder-only architectures (e.g., GPT), vision architectures (e.g., ViT: Vision Transformer), and more.

Also note that further tools used to improve GPTs after language model training:
* **Instruction tuning**: basically, provide correct answers to the question you asked (i.e., supervised learning).
* **Reinforcement learning from human feedback**: provide signal on whether the answer is good or bad (e.g., thumbs up or down). This is used as signal by the LLM to improve itself.

See [4] for more details on these aspects.

*This is a very quick introduction to the transformer, aiming only at providing intuition. In most cases, as we will see below, transformers are used as pre-trained models, that we then fine-tune or otherwise adjust to our specific tasks.*

In [ ]:
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertForSequenceClassification, DataCollatorWithPadding
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import numpy as np

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.colors import to_rgba
import seaborn as sns
sns.set_theme('notebook', style='whitegrid')

## Progress bar
from tqdm.notebook import tqdm

# Set device (GPU or CPU)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [ ]:
import torch
torch.manual_seed(42) # Setting the seed

print("Using torch", torch.__version__)

## Text classification

In this exercise, we are given a dataset of digitized books from the British Library. These books belong to different genres (e.g., poetry or prose). We are interested in training a classifier to distinguish between such genres. [Read more about this dataset here](https://github.com/mromanello/ADA-DHOxSS/tree/master/data#british-library-19th-century-books).

In [ ]:
import pandas as pd

In [ ]:
df_books = pd.read_csv('data/bl_books/sample_tidy/df_book.csv')
df_texts = pd.read_csv('data/bl_books/sample_tidy/df_book_text.csv')

In [ ]:
df_books.head(2)

In [ ]:
df_books.genre.value_counts()

In [ ]:
df = df_books.merge(df_texts, on='fulltext_filename')

In [ ]:
df.head(2)

In [ ]:
# divide all the data into training and testing

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
# Encode string labels into integers using LabelEncoder from sklearn
# The label column in both the training and testing dataframes contains string labels
label_encoder = LabelEncoder()
# Fit the encoder on the training labels and transform them to integers
train_df['label'] = label_encoder.fit_transform(train_df['genre'])
# Apply the same transformation on the test labels
test_df['label'] = label_encoder.transform(test_df['genre'])

In [ ]:
train_df.head(2)

In [ ]:
# Define a custom dataset class for our text data
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts = dataframe['fulltext'].values  # Get the 'text' column from the dataframe
        self.labels = dataframe['label'].values  # Get the 'label' column (which now contains integers)
        self.tokenizer = tokenizer  # BERT tokenizer passed as an argument
        self.max_len = max_len  # Maximum length for the tokenized input

    def __len__(self):
        return len(self.texts)  # Returns the number of samples in the dataset

    def __getitem__(self, index):
        text = self.texts[index]  # Get the text at the given index
        label = self.labels[index]  # Get the label corresponding to the text

        # Tokenize the text
        # encode_plus returns a dictionary with input_ids, attention_mask, etc.
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,  # Add [CLS] and [SEP] tokens
            max_length=self.max_len,  # Truncate or pad to the max length
            return_token_type_ids=False,  # We don't need token type IDs for classification
            padding='longest',  # Pad the sequence to max_len
            return_attention_mask=True,  # Return attention mask (which indicates padded tokens)
            return_tensors='pt',  # Return PyTorch tensors
            truncation=True  # Truncate longer sequences
        )

        # Return the input ids, attention mask, and label for this sample
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Convert label to tensor
        }

# Function to train the model
def train_model(model, data_loader, optimizer, device, scheduler, n_examples):
    model = model.train()  # Set the model to training mode
    total_loss = 0  # Initialize the total loss
    correct_predictions = 0  # Initialize correct predictions count

    for batch in data_loader:
        # Move the data to the device (GPU or CPU)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass through the model
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss  # Get the loss from the output
        logits = outputs.logits  # Get the raw output (logits)

        # Backward pass and optimization
        optimizer.zero_grad()  # Zero out the gradients
        loss.backward()  # Perform backpropagation
        optimizer.step()  # Update model parameters
        scheduler.step()  # Update the learning rate based on the scheduler

        total_loss += loss.item()  # Accumulate the total loss
        _, preds = torch.max(logits, dim=1)  # Get the predicted labels (the highest logit)
        correct_predictions += torch.sum(preds == labels)  # Count correct predictions

    return correct_predictions.float() / n_examples, total_loss / len(data_loader)  # Return accuracy and average loss

# Function to evaluate the model on the validation set
def eval_model(model, data_loader, device, n_examples):
    model = model.eval()  # Set the model to evaluation mode
    total_loss = 0  # Initialize the total loss
    correct_predictions = 0  # Initialize correct predictions count

    with torch.no_grad():  # Disable gradient calculation (we're not training)
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)  # Move data to the device
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss  # Get the loss
            logits = outputs.logits  # Get the raw output (logits)

            total_loss += loss.item()  # Accumulate the loss
            _, preds = torch.max(logits, dim=1)  # Get the predicted labels
            correct_predictions += torch.sum(preds == labels)  # Count correct predictions

    return correct_predictions.float() / n_examples, total_loss / len(data_loader)  # Return accuracy and average loss

In [ ]:
# Load pre-trained BERT tokenizer and model
PRE_TRAINED_MODEL_NAME = 'bert-base-uncased'  # Using the uncased version of BERT
tokenizer = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME, clean_up_tokenization_spaces=True)  # Load tokenizer

# Initialize a data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Hyperparameters
FREEZE = True  # Whether to freeze the BERT layers or not
MAX_LEN = 512  # Maximum length of input sequences
BATCH_SIZE = 16  # Batch size for training and evaluation
EPOCHS = 5  # Number of epochs to train the model
LEARNING_RATE = 2e-5  # Learning rate for the optimizer

# Assume train_df and test_df are the pandas DataFrames containing the dataset
# train_df = pd.DataFrame(...)  # DataFrame containing 'fulltext' and 'label'
# test_df = pd.DataFrame(...)   # DataFrame containing 'fulltext' and 'label'

# Create datasets for training and testing using the TextDataset class
train_dataset = TextDataset(train_df, tokenizer, MAX_LEN)
test_dataset = TextDataset(test_df, tokenizer, MAX_LEN)

# Create DataLoaders to feed data into the model in batches
train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)  # Shuffle the training data
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=data_collator)  # No need to shuffle the test data

# Initialize the pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained(PRE_TRAINED_MODEL_NAME, num_labels=len(label_encoder.classes_))
model = model.to(device)  # Move the model to the device (GPU/CPU)

# Ensure all layers are trainable
# NB this is costly in terms of memory and computation
if FREEZE:
    for p in model.bert.parameters():
        p.requires_grad = False
else:
    for p in model.parameters():
        p.requires_grad = True

# Define the optimizer and learning rate scheduler
total_steps = len(train_data_loader) * EPOCHS  # Total number of training steps
warmup_steps = int(0.1 * total_steps)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01) # AdamW optimizer (recommended for BERT)

scheduler = get_linear_schedule_with_warmup( # Learning rate scheduler
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Training loop
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')  # Print the current epoch
    print('-' * 10)

    # Train the model for one epoch
    train_acc, train_loss = train_model(model, train_data_loader, optimizer, device, scheduler, len(train_df))
    print(f'Train loss {train_loss} accuracy {train_acc}')  # Print training loss and accuracy

    # Evaluate the model on the validation set
    test_acc, test_loss = eval_model(model, test_data_loader, device, len(test_df))
    print(f'Test loss {test_loss} accuracy {test_acc}')  # Print validation loss and accuracy

In [ ]:
# Evaluation and metrics (after training is completed)
y_true = []  # List to store true labels
y_pred = []  # List to store predicted labels

model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Disable gradient calculation (not needed for inference)
    for batch in test_data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Forward pass
        _, preds = torch.max(outputs.logits, dim=1)  # Get the predicted labels

        y_true.extend(labels.cpu().numpy())  # Store true labels
        y_pred.extend(preds.cpu().numpy())  # Store predicted labels

# Print the classification report (precision, recall, F1-score)
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, zero_division=1))

### Exercise 1 (Easy): RoBERTa

The model clearly could use some improvement. One way it to switch to a better model architecture, which allows larger inputs to be processed. Consider RoBERTa:

```Python
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Change the model name to a RoBERTa pre-trained model
PRE_TRAINED_MODEL_NAME = 'roberta-base'  # You can use 'roberta-large' for a larger model

# Load the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(PRE_TRAINED_MODEL_NAME, num_labels=len(label_encoder.classes_))

# Move model to the appropriate device
model = model.to(device)
```

You can remove the line: ```return_token_type_ids=False``` since RoBERTa uses a Byte-pair encoding that does not include type IDs. How is the performance changing? How about the training time?

### Exercise 2 (Medium): Book type

Try using the `type` column instead of genre. Is it evenly distributed or not? How does the classifier perform with it?

### Exercise 3 (Easy): DistilBERT

Swap `bert-base-uncased` for `distilbert-base-uncased` (`DistilBertTokenizer`, `DistilBertForSequenceClassification`). Compare wall-clock training time and accuracy against the original BERT run.

*A related task, Named Entity Recognition on 19th-century place names (`data/topRes19th_v2`), is available as optional homework in the git history of this repository (see the version of this notebook before commit that introduced this restructuring).*

---

## Transformers for images

So far we have used the Transformer for text. The same architecture, with images cut into patches treated as "tokens", is also the backbone of most modern computer vision models. Before getting there, let's briefly recap **Convolutional Neural Networks (CNNs)**, the architecture that dominated computer vision before transformers arrived, since the two ideas are complementary and still often combined in practice.

References for this part:
* [Computer Vision for the Humanities: An Introduction to Deep Learning for Image Classification](https://programminghistorian.org/en/lessons/computer-vision-deep-learning-pt1) from the Programming Historian series.
* [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020) (the CLIP paper, Radford et al. 2021).

### A brief recap of CNNs

**Convolutional Neural Networks** apply small filters (kernels) that slide across an image, detecting local patterns such as edges and textures in early layers, and progressively more complex shapes and objects in deeper layers.

<img src="figures/Convolutional_neural_network.png" width="500px" heigth="500px">

*[From Wikipedia](https://en.wikipedia.org/wiki/Convolutional_neural_network)*.

**Pooling layers** reduce the spatial size of feature maps (e.g., max pooling keeps only the strongest activation in each patch), which reduces computation and adds some invariance to small shifts in the image.

<img src="figures/Max_pooling.png" width="500px" heigth="500px">

**ResNet** architectures stack many convolutional layers, using *skip connections* that let information (and gradients) bypass layers, which makes it possible to train much deeper networks without them degrading.

<img src="figures/res_net.png">

CNNs remain a strong, efficient choice for many image tasks, and are often used together with or as an alternative to transformer-based vision models.

### From CNNs to Vision Transformers

The **Vision Transformer (ViT)** applies the same Transformer architecture we used for text to images: an image is cut into fixed-size patches (e.g., 16x16 pixels), each patch is flattened and linearly projected into an embedding, exactly as a word is embedded, and the resulting sequence of "patch tokens" is fed through the same self-attention layers we implemented above. This means all the machinery (Q/K/V, multi-head attention, positional encodings) transfers directly from text to images.

### CLIP: connecting images and text

**CLIP (Contrastive Language-Image Pre-training)** takes this one step further: it trains an image encoder (often ViT-based) and a text encoder *together*, so that the embedding of an image and the embedding of a caption describing it end up close together in the same vector space, while embeddings of unrelated image-text pairs end up far apart. This is called **contrastive pre-training**, and it is trained on hundreds of millions of image-caption pairs scraped from the web.

Because images and text share an embedding space, CLIP can:
* **Classify images with no training data**, just by comparing an image's embedding to the embeddings of a handful of candidate text descriptions ("zero-shot classification").
* **Search a collection of images using free-text queries**, or vice versa.

This is especially useful for GLAM (Galleries, Libraries, Archives, Museums) collections, which are rarely accompanied by consistent, exhaustive labels: CLIP lets us query and organize collections using natural language, without first having to train a bespoke classifier for every possible category.

### CLIP in practice: newspaper advertisements

We use the same newspaper ads dataset from the Programming Historian lesson: 549 scanned advertisement images, each labelled as either `text-only` or `illustrations` (`data/newspaper_images/ads_data/`).

In [ ]:
import os
import pandas as pd
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

csv_file = "data/newspaper_images/ads_data/ads_upsampled_no_index.csv"
img_dir = "data/newspaper_images/ads_data/images"

annotations = pd.read_csv(csv_file)
annotations["path"] = annotations.iloc[:, 0].apply(lambda f: os.path.join(img_dir, f))
labels = sorted(annotations["label"].unique())
print(f"{len(annotations)} images, labels: {labels}")

images = [Image.open(p).convert("RGB") for p in annotations["path"]]

#### Zero-shot classification

We describe each label in a short natural-language prompt, embed the prompts and all images, and assign each image to whichever prompt its embedding is closest to. No training involved.

In [ ]:
import torch
from sklearn.metrics import classification_report, confusion_matrix

# One text prompt per label -- wording matters a lot for zero-shot accuracy, try changing it
label_to_prompt = {
    "text-only": "a scanned newspaper advertisement with only text, no pictures",
    "illustrations": "a scanned newspaper advertisement with a drawing or illustration",
}
prompts = [label_to_prompt[l] for l in labels]

with torch.no_grad():
    inputs = clip_processor(text=prompts, images=images, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    # outputs.logits_per_image: similarity of each image to each prompt
    probs = outputs.logits_per_image.softmax(dim=1)

predicted_labels = [labels[i] for i in probs.argmax(dim=1).tolist()]
true_labels = annotations["label"].tolist()

print(classification_report(true_labels, predicted_labels))
print("Confusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(true_labels, predicted_labels, labels=labels))

#### Text-to-image search

We embed every image once, then compare against a free-text query to find the closest matches — a simple form of semantic image search.

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    image_inputs = clip_processor(images=images, return_tensors="pt", padding=True)
    image_embeds = clip_model.get_image_features(**image_inputs)
    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)

def search(query, top_k=5):
    with torch.no_grad():
        text_inputs = clip_processor(text=[query], return_tensors="pt", padding=True)
        text_embed = clip_model.get_text_features(**text_inputs)
        text_embed = text_embed / text_embed.norm(dim=-1, keepdim=True)
    similarity = (image_embeds @ text_embed.T).squeeze(1)
    top_indices = similarity.topk(top_k).indices.tolist()

    fig, axes = plt.subplots(1, top_k, figsize=(3 * top_k, 3))
    for ax, idx in zip(axes, top_indices):
        ax.imshow(images[idx])
        ax.set_title(f"sim={similarity[idx]:.2f}")
        ax.axis("off")
    fig.suptitle(f'Query: "{query}"')
    plt.show()

search("a ship")
search("a hat")

#### Linear probe: transfer learning without a training loop

We already have `image_embeds` for every image (computed above). Instead of prompting CLIP in natural language, we can train a plain `LogisticRegression` classifier directly on these frozen embeddings — a **linear probe**. This is the same transfer-learning idea used in the ResNet fine-tuning approach from Notebook 4's earlier edition (freeze the backbone, train only a linear layer on top), but here it takes seconds on a CPU since we never touch the CLIP weights.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = image_embeds.numpy()
y = annotations["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

probe = LogisticRegression(max_iter=1000)
probe.fit(X_train, y_train)

print("Linear probe accuracy:", probe.score(X_test, y_test))
print(classification_report(y_test, probe.predict(X_test)))

---

### Exercise 1 (Easy)

Rewrite the prompts used for zero-shot classification and see how much accuracy changes. Try adding more context (e.g., mentioning "OCR", "scanned", "1890s") or trying several prompt variants per label and averaging their embeddings.

### Exercise 2 (Medium)

Use `data/newspaper_images/photo_data/multi_label.csv`, which has three possibly co-occurring labels (`human`, `landscape`, `structure`) instead of one. Adapt the zero-shot approach to a multi-label setting: score each label independently against a threshold, rather than picking a single best match.

### Exercise 3 (Hard)

Look at the [WikiArt dataset](https://github.com/cs-chan/ArtGAN/blob/master/WikiArt%20Dataset/README.md). Pick a task of your choice among artist, genre, and style classification. Try zero-shot CLIP first, then a linear probe, then compare against fine-tuning a pretrained CNN (e.g., ResNet-18) end to end. Which approach works best, and why?